In [12]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

In [ ]:
# import dataset
california_housing = fetch_california_housing(as_frame=True)
X_housing = california_housing.data
y_housing = california_housing.target
X_train, X_test, y_train, y_test = train_test_split(X_housing, y_housing, test_size=0.2, random_state=0)

In [7]:
# train 2 raw model
ridge_raw = Ridge(alpha=0).fit(X_train, y_train)
tree_raw = DecisionTreeRegressor(random_state=0, max_depth=6).fit(X_train, y_train)
ridge_raw = r2_score(y_test, ridge_raw.predict(X_test))
tree_raw = r2_score(y_test, tree_raw.predict(X_test))

In [ ]:
# do standarization
ss = StandardScaler().fit(X_train)
ridge_std = Ridge(alpha=0).fit(ss.transform(X_train), y_train)
tree_std = DecisionTreeRegressor(random_state=0, max_depth=6).fit(ss.transform(X_train), y_train)
ridge_std = r2_score(y_test, ridge_std.predict(ss.transform(X_test)))
tree_std = r2_score(y_test, tree_std.predict(ss.transform(X_test)))

In [13]:
# log1p on skewed columns, then standardized
skewed = ['MedInc', 'AveRooms', 'Population', 'AveOccup']
X_train_log, X_test_log = X_train.copy(), X_test.copy()
X_train_log[skewed] = np.log1p(X_train_log[skewed])
X_test_log[skewed] = np.log1p(X_test_log[skewed])
ss2 = StandardScaler().fit(X_train_log)
ridge_ls = Ridge(alpha=1.0).fit(ss2.transform(X_train_log), y_train)
tree_ls  = DecisionTreeRegressor(random_state=0, max_depth=6).fit(ss2.transform(X_train_log), y_train)
ridge_ls = r2_score(y_test, ridge_ls.predict(ss2.transform(X_test_log)))
tree_ls  = r2_score(y_test, tree_ls.predict(ss2.transform(X_test_log)))

pd.DataFrame({
    'Ridge test R²':        [ridge_raw, ridge_std, ridge_ls],
    'DecisionTree test R²': [tree_raw,  tree_std,  tree_ls],
}, index=['(2) raw', '(3) standardized', '(4) log + standardized']).round(3)

# Write down your observations:
# ridge is sensitive to scaling, the R2 score change every time we do scaling in dataset
# otherwise decision tree model not response to 2 type of this scaling, we see the R2 score does not change at al

,Ridge test R²,DecisionTree test R²
(2) raw,0.594,0.618
(3) standardized,0.594,0.618
(4) log + standardized,0.606,0.618
